# EM-DAT International Disaster Database — Ingest

Loads the raw EM-DAT export, cleans and normalises it, and saves a processed parquet for use
in the liability pipeline.

## Data Attribution

> EM-DAT: The Emergency Events Database — Université catholique de Louvain (UCL) — CRED,
> D. Guha-Sapir — www.emdat.be, Brussels, Belgium.

Data obtained under the [EM-DAT Data Use Agreement](https://www.emdat.be/terms-conditions).
**No redistribution.** The raw export and processed parquet are excluded from version control.
Academic/non-commercial use only.

## How to obtain the data

1. Register for a free account at https://www.emdat.be
2. Log in and navigate to the query builder
3. Export: Disaster Group = Natural, all countries, 1900–present, CSV format
4. Save to `data/raw/emdat/` (any filename ending in `.csv` or `.xlsx`)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

RAW  = Path('../../data/raw')
PROC = Path('../../data/processed')

EMDAT_RAW = RAW / 'emdat'

# Locate raw file — accept any CSV or Excel in data/raw/emdat/
candidates = list(EMDAT_RAW.glob('*.csv')) + list(EMDAT_RAW.glob('*.xlsx'))
if not candidates:
    raise FileNotFoundError(
        f'No EM-DAT file found in {EMDAT_RAW}.\n'
        'Register at emdat.be, export a CSV of natural disasters, '
        f'and save it to {EMDAT_RAW}/'
    )
RAW_FILE = sorted(candidates)[-1]  # use most recently named file if multiple
print(f'Found: {RAW_FILE.name}  ({RAW_FILE.stat().st_size/1e6:.1f} MB)')

## 1. Load raw file

In [ ]:
# EM-DAT CSV exports have 6 header rows of metadata before the column names.
# Try skiprows=6 first; fall back to auto-detection if the shape looks wrong.
def _load_raw(path: Path) -> pd.DataFrame:
    if path.suffix == '.xlsx':
        for skip in [6, 0, 1]:
            df = pd.read_excel(path, skiprows=skip)
            if 'Dis No' in df.columns or 'DisNo' in df.columns:
                return df
        return pd.read_excel(path)
    else:
        for skip in [6, 0, 1]:
            df = pd.read_csv(path, skiprows=skip, low_memory=False)
            if 'Dis No' in df.columns or 'DisNo' in df.columns:
                return df
        return pd.read_csv(path, low_memory=False)

raw = _load_raw(RAW_FILE)
print(f'Shape: {raw.shape}')
print(f'Columns: {list(raw.columns)}')

## 2. Explore schema

In [ ]:
print('Disaster types:')
type_col = next((c for c in raw.columns if 'disaster type' in c.lower() or c.lower() == 'disaster_type'), None)
if type_col:
    print(raw[type_col].value_counts().head(15).to_string())

print()
print('Damage columns present:')
dmg_cols = [c for c in raw.columns if 'damage' in c.lower() or 'loss' in c.lower()]
for c in dmg_cols:
    n_populated = raw[c].notna().sum()
    print(f'  {c!r}: {n_populated} non-null values')

print()
print('Sample row:')
print(raw.head(2).T)

## 3. Clean and normalise

In [ ]:
# ── Column name normalisation ──
# EM-DAT exports use various capitalisations and spacing conventions across versions.
# Build a canonical rename map by fuzzy-matching key column names.

def _find_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    cols_lower = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

RENAME = {
    _find_col(raw, ['Dis No', 'DisNo', 'dis_no']):                    'dis_no',
    _find_col(raw, ['Disaster Group', 'disaster_group']):              'disaster_group',
    _find_col(raw, ['Disaster Type', 'disaster_type']):                'disaster_type',
    _find_col(raw, ['Disaster Subtype', 'disaster_subtype']):          'disaster_subtype',
    _find_col(raw, ['Event Name', 'event_name']):                      'event_name',
    _find_col(raw, ['Country', 'country']):                            'country',
    _find_col(raw, ['ISO', 'ISO3', 'country_iso3']):                   'country_iso3',
    _find_col(raw, ['Start Year', 'start_year']):                      'start_year',
    _find_col(raw, ['Start Month', 'start_month']):                    'start_month',
    _find_col(raw, ['Start Day', 'start_day']):                        'start_day',
    _find_col(raw, ['End Year', 'end_year']):                          'end_year',
    _find_col(raw, ['End Month', 'end_month']):                        'end_month',
    _find_col(raw, ['End Day', 'end_day']):                            'end_day',
    _find_col(raw, ['Total Deaths', 'total_deaths']):                  'total_deaths',
    _find_col(raw, ['Total Affected', 'total_affected']):              'total_affected',
    _find_col(raw, ["Total Damage ('000 US$)", 'total_damage_000_usd',
                    'Total Damage (USD, original)', 'total_damages_000']):  'total_damage_kusd',
    _find_col(raw, ["Insured Damage ('000 US$)", 'insured_damage_000_usd',
                    'insured_damages_000']):                                 'insured_damage_kusd',
    _find_col(raw, ["Total Damage, Adjusted ('000 US$)",
                    'total_damage_adj_000_usd']):                            'total_damage_adj_kusd',
}
RENAME = {k: v for k, v in RENAME.items() if k is not None}
print('Column mapping:', RENAME)

df = raw.rename(columns=RENAME).copy()

# ── Filter to natural disasters ──
if 'disaster_group' in df.columns:
    df = df[df['disaster_group'].str.lower().str.contains('natural', na=False)]

# ── Damage: '000 US$ → full USD ──
for src, dst in [('total_damage_kusd', 'total_damage_usd'),
                 ('insured_damage_kusd', 'insured_damage_usd'),
                 ('total_damage_adj_kusd', 'total_damage_adj_usd')]:
    if src in df.columns:
        df[dst] = pd.to_numeric(df[src], errors='coerce') * 1000

# ── CPI adjustment to 2020 USD ──
# US CPI (annual average, index 1982-84=100) from BLS, hardcoded for 1990-2023.
# Source: U.S. Bureau of Labor Statistics, CPI-U All Items.
CPI_US = {
    1990:130.7,1991:136.2,1992:140.3,1993:144.5,1994:148.2,1995:152.4,
    1996:156.9,1997:160.5,1998:163.0,1999:166.6,2000:172.2,2001:177.1,
    2002:179.9,2003:184.0,2004:188.9,2005:195.3,2006:201.6,2007:207.3,
    2008:215.3,2009:214.5,2010:218.1,2011:224.9,2012:229.6,2013:233.0,
    2014:236.7,2015:237.0,2016:240.0,2017:245.1,2018:251.1,2019:255.7,
    2020:258.8,2021:270.9,2022:292.7,2023:304.7,
}
CPI_BASE = CPI_US[2020]  # 258.8

def cpi_adjust(nominal_usd: pd.Series, event_year: pd.Series) -> pd.Series:
    factors = event_year.map(lambda y: CPI_BASE / CPI_US.get(int(y), CPI_BASE) if pd.notna(y) else 1.0)
    return nominal_usd * factors

if 'total_damage_usd' in df.columns and 'start_year' in df.columns:
    # Prefer the pre-adjusted column if available; otherwise CPI-adjust nominal
    if 'total_damage_adj_usd' in df.columns:
        df['total_damage_usd_2020'] = df['total_damage_adj_usd']  # already adjusted by EM-DAT
        print('Using EM-DAT pre-adjusted damage column.')
    else:
        df['total_damage_usd_2020'] = cpi_adjust(df['total_damage_usd'], df['start_year'])
        print('CPI-adjusted nominal damages to 2020 USD.')

print(f'Cleaned shape: {df.shape}')
print(f'Damage column coverage (total_damage_usd): {df["total_damage_usd"].notna().sum()} / {len(df)}')

## 4. Save processed parquet

In [ ]:
out_path = PROC / 'emdat_disasters.parquet'
df.to_parquet(out_path, index=False)
print(f'Saved {len(df)} records to {out_path}')
print(f'Size: {out_path.stat().st_size/1e6:.2f} MB')
print('\nNote: this file is in .gitignore (EM-DAT Data Use Agreement — no redistribution)')

## 5. Black Summer validation

In [ ]:
# Look for Black Summer 2019-20 — AUS, Wildfire, 2019 or 2020
bs_mask = (
    (df.get('country_iso3', df.get('country', '')).str.upper() == 'AUS')
    & (df['start_year'].astype(str).isin(['2019', '2020']))
    & (df['disaster_type'].str.lower().str.contains('wild|fire|bush', na=False))
)
bs = df[bs_mask]
print(f'Black Summer candidate records: {len(bs)}')
if len(bs) > 0:
    display_cols = [c for c in ['dis_no','event_name','start_year','start_month',
                                 'disaster_type','total_deaths','total_damage_usd',
                                 'insured_damage_usd','total_damage_usd_2020']
                    if c in bs.columns]
    print(bs[display_cols].to_string(index=False))
    print()

    # Compare to hardcoded scenarios
    AUD_TO_USD = 0.69
    D_CENTRAL_MANUAL = 10.0 * AUD_TO_USD  # AUD 10B → USD 6.9B
    D_CONSERVATIVE_MANUAL = 2.32 * AUD_TO_USD  # AUD 2.32B → USD 1.6B

    for _, row in bs.iterrows():
        td = row.get('total_damage_usd', np.nan)
        ins = row.get('insured_damage_usd', np.nan)
        print(f"--- {row.get('dis_no','?')} {row.get('event_name','')} ---")
        if pd.notna(td):
            print(f'  EM-DAT total damage:     USD {td/1e9:.2f}B')
            print(f'  Manual central scenario: USD {D_CENTRAL_MANUAL:.2f}B')
            print(f'  Difference:              {abs(td/1e9 - D_CENTRAL_MANUAL)/D_CENTRAL_MANUAL*100:.1f}%')
        if pd.notna(ins):
            print(f'  EM-DAT insured:          USD {ins/1e9:.2f}B')
            print(f'  Manual conservative:     USD {D_CONSERVATIVE_MANUAL:.2f}B')
else:
    print('No matching records — check disaster_type values:')
    print(df[df.get('country_iso3', df.get('country', '')).str.upper() == 'AUS']['disaster_type'].value_counts().head(10))

## Key findings

*To be filled after execution.*